# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring task, not classification, clustering, or ranking. Each page gets its own independent number representing how much it under-captures clicks relative to peers at the same position tier — not a fixed group label, and not a rank that depends on comparison to the rest of a batch. A new page could be scored on its own, without needing the rest of the dataset present.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship
Starter data found. You're ready.


In [5]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The proxy target is ctr_gap_ratio = page's own CTR ÷ its position_tier's average CTR (computed only among pages with enough impressions to trust). A ratio below 1.0 means the page is under-capturing clicks relative to its peers; above 1.0 means it's over-capturing. This is built entirely from observed outcomes (actual clicks/impressions), not from any decision-bucket field like health_score or priority_score.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

visible = df[df["impressions_90d"] >= 100].copy()
tier_avg_ctr = visible.groupby("position_tier")["ctr"].transform("mean")
visible["ctr_gap_ratio"] = visible["ctr"] / tier_avg_ctr
visible[["content_id", "position_tier", "ctr", "ctr_gap_ratio"]].head(10)

,content_id,position_tier,ctr,ctr_gap_ratio
0,content_304f48230142,striking,0.76,2.971282
1,content_a1fb4e703a9e,page_3_5,0.05,0.351225
2,content_9aa793d4d895,page_3_5,0.09,0.632205
3,content_331d6c4de07b,page_1,0.49,1.381217
4,content_d99b7a2d90ca,page_3_5,0.13,0.913185
5,content_d4084a4bc775,page_1,0.03,0.084564
7,content_a63219c6e95a,page_3_5,0.06,0.421470
8,content_5e6c160719bc,page_3_5,0.09,0.632205
9,content_c27558df2b0c,page_1,0.16,0.451010
10,content_d8ee6cc6d642,top_3,1.55,4.638947


## 3. Success metric

*One metric you can defend. What number means 'good'?*

he metric is correlation between predicted and observed ctr_gap_ratio on held-out pages (e.g., Spearman correlation, since we care about relative ordering of under-performers, not exact values) — evaluated with a client-holdout split so no client's pages appear in both train and test.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from scipy.stats import spearmanr
# Once a model produces predicted_ctr_gap_ratio on a holdout set:
# corr, _ = spearmanr(y_true, y_pred)
# print(f"Spearman correlation: {corr:.3f}")
print("Metric defined: Spearman correlation between predicted and actual ctr_gap_ratio on client-holdout pages.")

Metric defined: Spearman correlation between predicted and actual ctr_gap_ratio on client-holdout pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (content_id), filtered to pages with impressions_90d >= 100 so the CTR ratio is statistically meaningful.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = visible[["content_id", "position_tier", "content_type", "impressions_90d", "ctr", "ctr_gap_ratio"]]
print(unit_of_analysis.shape[0], "rows — one row per page")
unit_of_analysis.head(10)

22006 rows — one row per page


,content_id,position_tier,content_type,impressions_90d,ctr,ctr_gap_ratio
0,content_304f48230142,striking,keyword article,3803,0.76,2.971282
1,content_a1fb4e703a9e,page_3_5,keyword article,15320,0.05,0.351225
2,content_9aa793d4d895,page_3_5,keyword article,12581,0.09,0.632205
3,content_331d6c4de07b,page_1,keyword article,11751,0.49,1.381217
4,content_d99b7a2d90ca,page_3_5,keyword article,19140,0.13,0.913185
5,content_d4084a4bc775,page_1,keyword article,3970,0.03,0.084564
7,content_a63219c6e95a,page_3_5,keyword article,1724,0.06,0.421470
8,content_5e6c160719bc,page_3_5,keyword article,32574,0.09,0.632205
9,content_c27558df2b0c,page_1,keyword article,1240,0.16,0.451010
10,content_d8ee6cc6d642,top_3,keyword article,20919,1.55,4.638947


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g., "flag any page with CTR below 0.10") ignores that CTR naturally varies by position tier, content type, and other factors — a page_1 page at 0.20 CTR might be badly under-capturing, while a deep page at 0.06 might be normal. ML can learn the combination of features (position, content type, word count, etc.) that predicts a page's expected CTR, producing a more accurate peer benchmark than one manually-picked threshold. The pattern is too multi-dimensional for a single if-statement to capture without either flagging too many normal pages or missing real under-performers.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [10]:
# Section 6 — self-check, run this before committing
assert "ctr_gap_ratio" in visible.columns, "target proxy missing"
assert (visible["impressions_90d"] >= 100).all(), "low-impression pages leaked into scoring set"
print("Self-check passed. No client names, URLs, or private queries appear above.")

Self-check passed. No client names, URLs, or private queries appear above.
